# Introduction
This notebook is used to fetch, plot, and analyze experiment results.

## 1. Initial Setup

In [1]:
print("Hello World")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Hello World
Free GPU Memory (GB): 39.3936


In [2]:
print("\n################################")
print("Setting up environment...")
print("################################\n")

import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2


################################
Setting up environment...
################################

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability
Initializing src package
Initializing src package


## 2. Fetch Results

In [3]:
import seml
import pandas as pd

db_collection = 'llama-pert-awq-bnb-hqq'
states = ["COMPLETED"]

# Get the results
all_results = seml.evaluation.get_results(db_collection, to_data_frame=True, states=states)

print(f"Length of all_results before deduplication: {len(all_results)}")

# Get the list of columns that start with 'config.'
config_columns = [col for col in all_results.columns if col.startswith('config.')]

# Drop duplicates based on config columns, keeping the last occurrence
all_results = all_results.drop_duplicates(subset=config_columns, keep='last')

print(f"Length of all_results after deduplication: {len(all_results)}")
print("Columns used for deduplication:")
print(config_columns)
print("\nAll columns in the dataframe:")
print(all_results.columns)
all_results.head()

Output()

Output()

Length of all_results before deduplication: 4800
Length of all_results after deduplication: 4800
Columns used for deduplication:
['config.overwrite', 'config.db_collection', 'config.batch_size', 'config.dataset_name', 'config.device', 'config.exp_id', 'config.max_entries', 'config.max_new_tokens', 'config.model_name', 'config.n_beams', 'config.n_repeats', 'config.num_excel_rows', 'config.save_excel', 'config.seed', 'config.strategy', 'config.temperature', 'config.typo_intensity', 'config.typo_type', 'config.use_beam_search']

All columns in the dataframe:
Index(['_id', 'config.overwrite', 'config.db_collection', 'config.batch_size',
       'config.dataset_name', 'config.device', 'config.exp_id',
       'config.max_entries', 'config.max_new_tokens', 'config.model_name',
       'config.n_beams', 'config.n_repeats', 'config.num_excel_rows',
       'config.save_excel', 'config.seed', 'config.strategy',
       'config.temperature', 'config.typo_intensity', 'config.typo_type',
       'config

,_id,config.overwrite,config.db_collection,config.batch_size,config.dataset_name,config.device,config.exp_id,config.max_entries,config.max_new_tokens,config.model_name,...,result.Brier_adj,result.LogLoss_adj,result.Entropy_adj,result.AUCROC_sem,result.AUCPR_sem,result.Brier_sem,result.LogLoss_sem,result.Entropy_sem,result.Accuracy,result.fail_trace
0,1,1,llama-pert-awq-bnb-hqq,32,P101,cuda,pert-awq-bnb-hqq-10-07,None,25,Llama-3-8B,...,0.561755,5.918077,36.896553,1.0,1.0,0.0,2.220446e-16,-3.981839e-08,0.396552,<function get_results at 0x7f842c07c280>
1,2,2,llama-pert-awq-bnb-hqq,32,P101,cuda,pert-awq-bnb-hqq-10-07,None,25,Llama-3-8B,...,0.561755,5.918077,36.896553,1.0,1.0,0.0,2.220446e-16,-3.981839e-08,0.396552,<function get_results at 0x7f842c07c280>
2,3,3,llama-pert-awq-bnb-hqq,32,P101,cuda,pert-awq-bnb-hqq-10-07,None,25,Llama-3-8B,...,0.561755,5.918077,36.896553,1.0,1.0,0.0,2.220446e-16,-3.981839e-08,0.396552,<function get_results at 0x7f842c07c280>
3,4,4,llama-pert-awq-bnb-hqq,32,P101,cuda,pert-awq-bnb-hqq-10-07,None,25,Llama-3-8B,...,0.579266,5.823508,45.919863,1.0,1.0,0.0,2.220446e-16,-3.765434e-08,0.375000,<function get_results at 0x7f842c07c280>
4,5,5,llama-pert-awq-bnb-hqq,32,P101,cuda,pert-awq-bnb-hqq-10-07,None,25,Llama-3-8B,...,0.624608,6.945512,41.225135,1.0,1.0,0.0,2.220446e-16,-3.246064e-08,0.323276,<function get_results at 0x7f842c07c280>


### 2.1 Check Failed Rows

In [8]:
import seml
import pandas as pd
from collections import Counter

db_collection = 'llama-pert-awq-bnb-hqq'
states = ["FAILED"]

# Get the results
failed_results = seml.evaluation.get_results(db_collection, to_data_frame=True, states=states)
print(f"Length of failed_results before deduplication: {len(failed_results)}")

# Get the list of columns that start with 'config.'
config_columns = [col for col in failed_results.columns if col.startswith('config.')]

# Drop duplicates based on config columns, keeping the last occurrence
failed_results = failed_results.drop_duplicates(subset=config_columns, keep='last')
print(f"Length of failed_results after deduplication: {len(failed_results)}")

print("\nUnique values and their frequencies for each config parameter in failed_results:")
for col in ["config.dataset_name", "config.model_name", "config.typo_type", "config.typo_intensity"]:
    value_counts = Counter(failed_results[col])
    print(f"\n{col}:")
    for value, count in value_counts.items():
        print(f"  - {value}: {count}")

Output()

Output()

Length of failed_results before deduplication: 70
Length of failed_results after deduplication: 70

Unique values and their frequencies for each config parameter in failed_results:

config.dataset_name:
  - P364: 11
  - P37: 18
  - P740: 21
  - P101: 20

config.model_name:
  - Llama-3-8B-BNB-4bit-local: 50
  - Llama-3-8B-HQQ-mixed-local: 20

config.typo_type:
  - word_phrase_translation: 2
  - word_context_aware_insertion: 1
  - word_remove_punctuation: 5
  - word_keyword_only: 5
  - word_taxonomy_pos: 6
  - word_taxonomy_neg: 5
  - none: 5
  - char_insertion: 5
  - char_deletion: 6
  - char_replacement: 4
  - char_repetition: 5
  - char_swapping: 3
  - word_CMW: 4
  - char_LCC: 5
  - word_synonym: 1
  - char_insert_noise: 3
  - word_repeat: 1
  - char_substitution: 3
  - word_emoji: 1

config.typo_intensity:
  - 1: 24
  - 2: 23
  - 3: 23


### 2.2 Check high accuracy columns

In [9]:
# Filter rows where accuracy is higher than 0.6
high_accuracy_results = all_results[all_results['result.Accuracy'] > 0.6]

# Print the number of rows that meet this criteria
print(f"Number of rows with accuracy > 0.6: {len(high_accuracy_results)}")

# Display the first few rows of the filtered results
print(high_accuracy_results[['config.model_name', 'config.strategy', 'result.Accuracy']].head())

# Count the number of high accuracy rows for each unique model name
model_counts = high_accuracy_results['config.model_name'].value_counts()

print("\nNumber of high accuracy rows for each model:")
print(model_counts)

# Optional: Calculate and print the percentage of high accuracy rows for each model
total_rows = len(all_results)
model_percentages = (model_counts / total_rows * 100).round(2)

print("\nPercentage of high accuracy rows for each model:")
print(model_percentages)

Number of rows with accuracy > 0.6: 2656
   config.model_name    config.strategy  result.Accuracy
54        Llama-3-8B  Direct Completion         0.669540
55        Llama-3-8B  Direct Completion         0.744253
56        Llama-3-8B  Direct Completion         0.778736
60        Llama-3-8B  Direct Completion         0.939611
61        Llama-3-8B  Direct Completion         0.939611

Number of high accuracy rows for each model:
config.model_name
Llama-3-8B                    756
Llama-3-8B-AWQ-4bit-local     714
Llama-3-8B-BNB-4bit-local     620
Llama-3-8B-HQQ-mixed-local    566
Name: count, dtype: int64

Percentage of high accuracy rows for each model:
config.model_name
Llama-3-8B                    15.75
Llama-3-8B-AWQ-4bit-local     14.88
Llama-3-8B-BNB-4bit-local     12.92
Llama-3-8B-HQQ-mixed-local    11.79
Name: count, dtype: float64


## 3. Plot results

In [4]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
from fpdf import FPDF
import numpy as np
from PIL import Image

# Add these imports at the top
import matplotlib.pyplot as plt

# import matplotlib
# matplotlib.rcParams['text.usetex'] = True

# Replace with:
import matplotlib
matplotlib.rcParams['text.usetex'] = False

# If you still see font-related issues, you might also want to add:
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

from src.models import get_model_num_bits

# Define color scheme for the four models
model_colors = {
    'Llama-3-8B': '#1f77b4',  # Blue
    'Llama-3-8B-AWQ-4bit-local': '#ff7f0e',  # Orange
    'Llama-3-8B-BNB-4bit-local': '#2ca02c',  # Green
    'Llama-3-8B-HQQ-mixed-local': '#d62728'  # Red
}

# Define color scheme for intensities
intensity_colors = {
    1: '#1f77b4',  # Blue
    2: '#2ca02c',  # Green
    3: '#d62728'   # Red
}

model_name_map = {
    'Llama-3-8B': 'Llama-3-8B',
    'Llama-3-8B-AWQ-4bit-local': 'Llama-3-8B-AWQ-4bit',
    'Llama-3-8B-BNB-4bit-local': 'Llama-3-8B-BNB-4bit',
    'Llama-3-8B-HQQ-mixed-local': 'Llama-3-8B-HQQ-3-4bit'
}

/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/fpdf/__init__.py:39: UserWarning: You have both PyFPDF & fpdf2 installed. Both packages cannot be installed at the same time as they share the same module namespace. To only keep fpdf2, run: pip uninstall --yes pypdf && pip install --upgrade fpdf2
  warnings.warn(


## 3.1 Radar plots

In [16]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import os

def generate_radar_plots(all_results, exp_id, plots_dir, model_colors, model_name_map):
    plots = []
    metrics = ['Accuracy', 'AUCPR_sample']
    
    for metric in metrics:
        metric_plots = []
        for intensity in [1, 2, 3]:
            fig = make_subplots(
                rows=3, cols=1, 
                specs=[[{'type': 'polar'}]]*3,
                subplot_titles=["Llama vs AWQ", "Llama vs BNB", "Llama vs HQQ"],
                vertical_spacing=0.3
            )
            
            df_intensity = all_results[all_results['config.typo_intensity'] == intensity]
            base_model = 'Llama-3-8B'
            comparison_models = [
                'Llama-3-8B-AWQ-4bit-local', 
                'Llama-3-8B-BNB-4bit-local', 
                'Llama-3-8B-HQQ-mixed-local'
            ]
            
            baseline_value = all_results[
                (all_results['config.typo_type'] == 'none') & 
                (all_results['config.model_name'] == base_model)
            ][f'result.{metric}'].mean()
            
            for i, comp_model in enumerate(comparison_models):
                for model in [base_model, comp_model]:
                    df_model = df_intensity[df_intensity['config.model_name'] == model]
                    
                    if not df_model.empty:
                        df_avg = df_model.groupby('config.typo_type')[f'result.{metric}'].mean().reset_index()
                        values = df_avg[f'result.{metric}'].tolist()
                        
                        if values:
                            values.append(values[0])
                            theta = df_avg['config.typo_type'].tolist()
                            theta.append(theta[0])
                            
                            fig.add_trace(
                                go.Scatterpolar(
                                    r=values,
                                    theta=theta,
                                    fill='toself',
                                    name=model_name_map[model],
                                    line=dict(color=model_colors[model])
                                ), 
                                row=i+1, 
                                col=1
                            )
                
                # Add baseline circle
                theta = np.linspace(0, 360, 100)
                fig.add_trace(
                    go.Scatterpolar(
                        r=[baseline_value] * len(theta),
                        theta=theta,
                        name='Baseline',
                        line=dict(color='gray', dash='dash'),
                        showlegend=(i == 0)
                    ), 
                    row=i+1, 
                    col=1
                )
            
            fig.update_layout(
                height=1800,
                width=800,
                title=f'Comparison of Average {metric} for Intensity {intensity} Across All Datasets',
                font=dict(size=10),
            )
            
            for i in range(1, 4):
                fig.update_layout(**{
                    f'polar{i}': dict(
                        radialaxis=dict(visible=True, range=[0, 1]),
                        angularaxis=dict(
                            tickfont=dict(size=8),
                            rotation=90,
                            direction="clockwise"
                        )
                    )
                })
            
            plot_file = os.path.join(
                plots_dir, 
                f"{metric}_Radar_Plot_Intensity_{intensity}_{exp_id}.png"
            )
            fig.write_image(plot_file, scale=2)
            metric_plots.append(
                (plot_file, f"Comparison Radar Plot of {metric} for Intensity {intensity}")
            )
            
        plots.append((metric, 'radar_plots', metric_plots))
    
    return plots

## 3.2 Box plots

In [17]:
import plotly.graph_objects as go
import os

def generate_boxplots(all_results, exp_id, plots_dir, model_colors, get_model_num_bits):
    plots = []
    metrics = ['Accuracy', 'AUCPR_sample']
    
    for metric in metrics:
        metric_plots = []
        fig = go.Figure()
        
        for model in model_colors.keys():
            num_bits = get_model_num_bits(model)
            for intensity in [1, 2, 3]:
                df_subset = all_results[
                    (all_results['config.model_name'] == model) & 
                    (all_results['config.typo_intensity'] == intensity)
                ]
                
                fig.add_trace(
                    go.Box(
                        y=df_subset[f'result.{metric}'],
                        x=df_subset['config.typo_intensity'],
                        name=f'{model} ({num_bits}-bit, Intensity {intensity})',
                        marker_color=model_colors[model],
                        showlegend=intensity == 1
                    )
                )
        
        fig.update_layout(
            title=f'Distribution of {metric} across Intensities and Models',
            xaxis_title='Intensity',
            yaxis_title=metric,
            boxmode='group',
            height=600,
            width=1200,
        )
        
        plot_file = os.path.join(
            plots_dir, 
            f"{metric}_Boxplot_Comparison_{exp_id}.png"
        )
        fig.write_image(plot_file, scale=2)
        metric_plots.append((plot_file, f"Boxplot Comparison of {metric}"))
        
        plots.append((metric, 'boxplots', metric_plots))
    
    return plots

## 3.3 Bar plots

In [18]:
import plotly.graph_objects as go
import os

def generate_bar_plots(all_results, exp_id, plots_dir, model_colors, get_model_num_bits):
    plots = []
    metrics = ['Accuracy', 'AUCPR_sample']
    
    intensity_colors = {
        1: '#1f77b4',  # Blue
        2: '#2ca02c',  # Green
        3: '#d62728'   # Red
    }
    
    for metric in metrics:
        metric_plots = []
        
        for model in model_colors.keys():
            baseline_value = all_results[
                (all_results['config.typo_type'] == 'none') & 
                (all_results['config.model_name'] == model)
            ][f'result.{metric}'].mean()
            
            perturbation_types = all_results['config.typo_type'].unique()
            perturbation_types = [p for p in perturbation_types if p != 'none']
            
            fig = go.Figure()
            
            for intensity in [1, 2, 3]:
                y = []
                for pert_type in perturbation_types:
                    value = all_results[
                        (all_results['config.typo_type'] == pert_type) & 
                        (all_results['config.typo_intensity'] == intensity) & 
                        (all_results['config.model_name'] == model)
                    ][f'result.{metric}'].mean()
                    y.append(value)
                
                fig.add_trace(
                    go.Bar(
                        x=perturbation_types,
                        y=y,
                        name=f'Intensity {intensity}',
                        marker_color=intensity_colors[intensity]
                    )
                )
            
            fig.add_shape(
                type="line",
                x0=-0.5,
                x1=len(perturbation_types)-0.5,
                y0=baseline_value,
                y1=baseline_value,
                line=dict(color="black", width=2, dash="dash")
            )
            
            num_bits = get_model_num_bits(model)
            fig.update_layout(
                title=f'All Perturbations Bar Plot of {metric} for {model} ({num_bits}-bit)',
                xaxis_title="Perturbation Type",
                yaxis_title=metric,
                barmode='group',
                height=600,
                width=1200
            )
            fig.update_xaxes(tickangle=45)
            
            plot_file = os.path.join(
                plots_dir, 
                f"{metric}_All_Perturbations_Bar_Plot_{model}_{exp_id}.png"
            )
            fig.write_image(plot_file, scale=2)
            metric_plots.append(
                (plot_file, f"All Perturbations Bar Plot of {metric} for {model}")
            )
        
        plots.append((metric, 'bar_plots', metric_plots))
    
    return plots

## 3.4 PDF Visualization

In [ ]:
from fpdf import FPDF
import matplotlib.pyplot as plt
import os
from PIL import Image

class MultipleVisualizationPDFGenerator:
    def __init__(self, plots_dir, exp_id):
        self.plots_dir = plots_dir
        self.exp_id = exp_id
        
    class CustomPDF(FPDF):
        def __init__(self, metric):
            super().__init__()
            self.metric = metric
            
        def footer(self):
            self.set_y(-15)
            self.set_font('Helvetica', 'I', 8)
            self.cell(0, 10, f'Page {self.page_no()}', 0, 0, 'C')
            
            if self.page_no() > 1:  # Skip title page
                if self.metric in ["Accuracy", "AUCPR_sample"]:
                    plt.figure(figsize=(6, 1))
                    plt.axis('off')
                    if self.metric == "Accuracy":
                        plt.text(0.5, 0.5, r'$\mathrm{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$', 
                                fontsize=12, ha='center')
                    else:
                        plt.text(0.5, 0.5, r'$\mathrm{AUCPR} = \int_0^1 \frac{TP}{TP + FP} d(\frac{TP}{TP + FN})$', 
                                fontsize=12, ha='center')
                    
                    eq_file = f'temp_equation_{self.page_no()}.png'
                    plt.savefig(eq_file, bbox_inches='tight', dpi=300, transparent=True)
                    plt.close()
                    
                    self.image(eq_file, x=60, y=270, w=90)
                    os.remove(eq_file)
    
    def generate_config_pages(self, pdf, all_results, plot_type):
        # Add title page
        pdf.add_page()
        pdf.set_font("Helvetica", 'B', size=14)
        pdf.cell(0, 10, f"Model Comparison: {plot_type} Analysis for {pdf.metric}", ln=True, align='C')
        
        # Add configuration page
        pdf.add_page()
        pdf.set_font("Helvetica", 'B', size=14)
        pdf.cell(0, 10, "Configuration Details", ln=True)
        pdf.set_font("Helvetica", size=8)
        
        config_columns = [
            "batch_size", "dataset_name", "exp_id", "max_entries",
            "max_new_tokens", "model_name", "n_beams", "n_repeats",
            "strategy", "temperature", "typo_intensity", "typo_type",
            "use_beam_search"
        ]
        
        for col in config_columns:
            unique_values = all_results[f"config.{col}"].unique()
            text = f"{col}: {', '.join(map(str, unique_values))}"
            text_width = pdf.get_string_width(text)
            if text_width > pdf.w - 2*pdf.l_margin:
                pdf.multi_cell(0, 5, text)
            else:
                pdf.cell(0, 5, text, ln=True)
    
    def create_pdf_for_plots(self, plots, metric, plot_type, all_results):
        pdf = self.CustomPDF(metric)
        pdf.set_auto_page_break(auto=True, margin=25)
        
        # Generate title and config pages
        self.generate_config_pages(pdf, all_results, plot_type)
        
        # Add plots
        for plot_file, desc in plots:
            pdf.add_page()
            pdf.set_font("Helvetica", 'B', size=14)
            pdf.cell(0, 20, desc, ln=True, align='C')
            
            with Image.open(plot_file) as img:
                img_width, img_height = img.size
            
            scale_factor = (pdf.w - 20) / img_width
            scaled_height = img_height * scale_factor
            
            pdf.image(plot_file, x=10, y=pdf.get_y(), w=pdf.w-20, h=scaled_height)
        
        # Save PDF
        output_path = os.path.join(
            self.plots_dir, 
            f"model_comparison_{metric}_{plot_type}_{self.exp_id}.pdf"
        )
        pdf.output(output_path)
        print(f"Generated PDF: {output_path}")
    
    def generate_all_pdfs(self, all_plot_data, all_results):
        """
        all_plot_data should be a list of tuples (metric, plot_type, plots)
        where plots is a list of (plot_file, description) tuples
        """
        for metric, plot_type, plots in all_plot_data:
            self.create_pdf_for_plots(plots, metric, plot_type, all_results)

# Example usage in main:
def main(all_results, exp_id):
    plots_dir = f"plots/model_comparison_{exp_id}"
    os.makedirs(plots_dir, exist_ok=True)
    
    # Initialize plot generators (using your existing functions)
    radar_plots = generate_radar_plots(all_results, exp_id, plots_dir, model_colors, model_name_map)
    box_plots = generate_boxplots(all_results, exp_id, plots_dir, model_colors, get_model_num_bits)
    bar_plots = generate_bar_plots(all_results, exp_id, plots_dir, model_colors, get_model_num_bits)
    
    # Combine all plots
    all_plot_data = radar_plots + box_plots + bar_plots
    
    # Initialize PDF generator and create PDFs
    pdf_generator = MultipleVisualizationPDFGenerator(plots_dir, exp_id)
    pdf_generator.generate_all_pdfs(all_plot_data, all_results)

# Sample execution:
if __name__ == "__main__":
    # Specify the experiment ID
    exp_id = "awq_hqq_bnb_comparison-10-19"
    
    # Call the main function with your dataframe
    main(all_results, exp_id)

/tmp/ipykernel_341002/361752375.py:43: DeprecationWarning:

The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.

/tmp/ipykernel_341002/361752375.py:19: DeprecationWarning:

The parameter "ln" is deprecated since v2.5.2. Instead of ln=0 use new_x=XPos.RIGHT, new_y=YPos.TOP.

/tmp/ipykernel_341002/361752375.py:48: DeprecationWarning:

The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.

/tmp/ipykernel_341002/361752375.py:65: DeprecationWarning:

The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.

/tmp/ipykernel_341002/361752375.py:78: DeprecationWarning:

The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.



Generated PDF: plots/model_comparison_awq_hqq_bnb_comparison-10-19/model_comparison_Accuracy_radar_plots_awq_hqq_bnb_comparison-10-19.pdf
Generated PDF: plots/model_comparison_awq_hqq_bnb_comparison-10-19/model_comparison_AUCPR_sample_radar_plots_awq_hqq_bnb_comparison-10-19.pdf
Generated PDF: plots/model_comparison_awq_hqq_bnb_comparison-10-19/model_comparison_Accuracy_boxplots_awq_hqq_bnb_comparison-10-19.pdf


## 4. Debug HQQ

In [8]:
import pandas as pd
import numpy as np

# Parameters for filtering
metric = 'Accuracy'  # or 'AUCPR_sample'
model = 'Llama-3-8B-HQQ-mixed-local'  # adjust as needed

# Get the exact filtered DataFrame we're interested in
filter_condition = (all_results['config.typo_type'] == 'none') & (all_results['config.model_name'] == model)
filtered_df = all_results[filter_condition]

# Print information about the filtering
print(f"\nFiltering for:")
print(f"config.typo_type == 'none' AND config.model_name == '{model}'")

print(f"\nNumber of rows found: {len(filtered_df)}")

# Print the complete filtered DataFrame
print("\nComplete filtered DataFrame:")
print(filtered_df)

# Print the specific columns we're most interested in
columns_of_interest = ['config.typo_type', 'config.model_name', f'result.{metric}']
print(f"\nFiltered DataFrame (key columns only):")
print(filtered_df[columns_of_interest])

# Print the actual baseline value that would be used
if not filtered_df.empty:
    baseline_value = filtered_df[f'result.{metric}'].iloc[0]
    print(f"\nBaseline value that would be used: {baseline_value}")
    
    # Additional validation
    if len(filtered_df) > 1:
        print("\nWARNING: Multiple rows found! All values:")
        print(filtered_df[f'result.{metric}'].values)
else:
    print(f"\nWARNING: No data found for model '{model}' with typo_type 'none'")

# Print unique values in key columns to help with debugging
print("\nUnique values in key columns:")
print("\nUnique typo_types:")
print(all_results['config.typo_type'].unique())
print("\nUnique model_names:")
print(all_results['config.model_name'].unique())


Filtering for:
config.typo_type == 'none' AND config.model_name == 'Llama-3-8B-HQQ-mixed-local'

Number of rows found: 60

Complete filtered DataFrame:
       _id  config.overwrite    config.db_collection  config.batch_size  \
3550  3603              3603  llama-pert-awq-bnb-hqq                 32   
3590  3661              3661  llama-pert-awq-bnb-hqq                 32   
3591  3662              3662  llama-pert-awq-bnb-hqq                 32   
3592  3663              3663  llama-pert-awq-bnb-hqq                 32   
3650  3721              3721  llama-pert-awq-bnb-hqq                 32   
3651  3722              3722  llama-pert-awq-bnb-hqq                 32   
3652  3723              3723  llama-pert-awq-bnb-hqq                 32   
3710  3781              3781  llama-pert-awq-bnb-hqq                 32   
3711  3782              3782  llama-pert-awq-bnb-hqq                 32   
3712  3783              3783  llama-pert-awq-bnb-hqq                 32   
3770  3841            

In [10]:
all_results["config.model_name"].unique()

array(['Llama-3-8B', 'Llama-3-8B-AWQ-4bit-local',
       'Llama-3-8B-BNB-4bit-local', 'Llama-3-8B-HQQ-mixed-local'],
      dtype=object)

In [9]:
import pandas as pd
import numpy as np

# Models to compare
hqq_model = 'Llama-3-8B-HQQ-mixed-local'
base_model = 'Llama-3-8B'
metric = 'Accuracy'

# Get filtered DataFrames for both models
hqq_filter = (all_results['config.typo_type'] == 'none') & (all_results['config.model_name'] == hqq_model)
base_filter = (all_results['config.typo_type'] == 'none') & (all_results['config.model_name'] == base_model)

hqq_df = all_results[hqq_filter]
base_df = all_results[base_filter]

# Print side-by-side comparison
print("\n=== Model Comparison (Baseline Accuracy) ===")
print("-" * 50)
print(f"HQQ Model: {hqq_model}")
print(f"Base Model: {base_model}")
print("-" * 50)

if not hqq_df.empty and not base_df.empty:
    hqq_accuracy = hqq_df[f'result.{metric}'].iloc[0]
    base_accuracy = base_df[f'result.{metric}'].iloc[0]
    
    print(f"\nAccuracy Values:")
    print(f"{'Model':<30} {'Accuracy':<10}")
    print("-" * 40)
    print(f"{hqq_model:<30} {hqq_accuracy:.4f}")
    print(f"{base_model:<30} {base_accuracy:.4f}")
    
    # Calculate difference
    diff = hqq_accuracy - base_accuracy
    print(f"\nDifference (HQQ - Base): {diff:.4f}")
    print(f"Relative Change: {(diff/base_accuracy)*100:.2f}%")
    
    # Print warning if multiple rows found
    if len(hqq_df) > 1:
        print(f"\nWARNING: Multiple rows found for HQQ model! All values:")
        print(hqq_df[f'result.{metric}'].values)
    if len(base_df) > 1:
        print(f"\nWARNING: Multiple rows found for Base model! All values:")
        print(base_df[f'result.{metric}'].values)
else:
    if hqq_df.empty:
        print(f"No data found for HQQ model with typo_type 'none'")
    if base_df.empty:
        print(f"No data found for Base model with typo_type 'none'")

# Print full filtered DataFrames for verification
print("\n=== Full Filtered DataFrames ===")
print("\nHQQ Model DataFrame:")
print(hqq_df[['config.typo_type', 'config.model_name', f'result.{metric}']])
print("\nBase Model DataFrame:")
print(base_df[['config.typo_type', 'config.model_name', f'result.{metric}']])


=== Model Comparison (Baseline Accuracy) ===
--------------------------------------------------
HQQ Model: Llama-3-8B-HQQ-mixed-local
Base Model: Llama-3-8B
--------------------------------------------------

Accuracy Values:
Model                          Accuracy  
----------------------------------------
Llama-3-8B-HQQ-mixed-local     0.3448
Llama-3-8B                     0.3966

Difference (HQQ - Base): -0.0517
Relative Change: -13.04%

[0.34482759 0.87819857 0.87819857 0.87819857 0.4386423  0.4386423
 0.4386423  0.7026087  0.7026087  0.7026087  0.71794872 0.72649573
 0.72649573 0.80701754 0.80701754 0.80701754 0.67321613 0.67218201
 0.67218201 0.6827957  0.6827957  0.6827957  0.92464358 0.92464358
 0.92464358 0.90709459 0.90709459 0.90709459 0.25529661 0.25529661
 0.25529661 0.36621196 0.36621196 0.36621196 0.29370629 0.3030303
 0.3030303  0.71221532 0.71221532 0.71221532 0.6903024  0.6903024
 0.6903024  0.77128205 0.77128205 0.77128205 0.64953271 0.64953271
 0.64953271 0.7049689

## 5. Remove Duplicates

In [5]:
import pandas as pd

def inspect_and_remove_duplicates(df):
    # Define the columns used for pivoting
    pivot_columns = ['config.typo_type', 'config.typo_intensity']
    
    # Find duplicates in pivot columns
    duplicate_mask = df.duplicated(subset=pivot_columns, keep=False)
    duplicates = df[duplicate_mask]
    
    if duplicates.empty:
        print("No duplicates found in pivot columns.")
        return df
    
    print("Duplicate entries found in pivot columns:")
    print(duplicates[pivot_columns])
    
    print("\nFull rows for duplicate entries:")
    print(duplicates)
    
    # Ask user how to handle duplicates
    print("\nHow would you like to handle these duplicates?")
    print("1: Keep first occurrence")
    print("2: Keep last occurrence")
    print("3: Remove all duplicates")
    print("4: Do nothing (keep all)")
    
    choice = input("Enter your choice (1-4): ")
    
    if choice == '1':
        df_cleaned = df.drop_duplicates(subset=pivot_columns, keep='first')
        print(f"Removed {len(df) - len(df_cleaned)} duplicate rows.")
    elif choice == '2':
        df_cleaned = df.drop_duplicates(subset=pivot_columns, keep='last')
        print(f"Removed {len(df) - len(df_cleaned)} duplicate rows.")
    elif choice == '3':
        df_cleaned = df.drop_duplicates(subset=pivot_columns, keep=False)
        print(f"Removed {len(df) - len(df_cleaned)} duplicate rows.")
    elif choice == '4':
        df_cleaned = df
        print("No rows removed.")
    else:
        print("Invalid choice. No rows removed.")
        df_cleaned = df
    
    return df_cleaned

# Assuming your dataframe is named 'all_results'
all_results_llama = inspect_and_remove_duplicates(all_results_llama)

# You can now use all_results_cleaned for further processing

Duplicate entries found in pivot columns:
           config.typo_type  config.typo_intensity
42  word_phrase_translation                      1
43  word_phrase_translation                      2
59  word_phrase_translation                      1
60  word_phrase_translation                      2

Full rows for duplicate entries:
    _id  config.overwrite config.db_collection config.dataset_name  \
42   43                43      llama-typo-eval                 P17   
43   44                44      llama-typo-eval                 P17   
59   61                61      llama-typo-eval                 P17   
60   62                62      llama-typo-eval                 P17   

   config.device    config.exp_id config.max_entries  config.max_new_tokens  \
42          cuda  typo-test-10-01               None                     25   
43          cuda  typo-test-10-01               None                     25   
59          cuda  typo-test-10-01               None                     25   
60